# Astrocalibration — PlantCV Pro pipeline

Robust, automated colour-calibration + phenotyping for the **AstroBotany "Astrocalibration" marker**
(ASOP-000x / AIRI Bio-Imaging Spectrum), the Python counterpart to the browser tool at
<https://dr-richard-barker.github.io/Anthocyanin-Image-analysis/>.

It uses **PlantCV + OpenCV** for the parts that are hard to do reliably in a browser:

1. **Detect the marker** (colour-card auto-detect, with a geometric fiducial-square fallback).
2. **Colour-correct** via an affine transform to the PlantCV `astro_color_matrix()` standard.
3. **Set scale** (pixels -> cm) from the marker.
4. **Segment** the plant (Excess-Green) and **quantify** leaf area + indices (NGRDI, mACI, GI).
5. **Export** a per-image CSV and the corrected image.

> Runs on Google Colab or locally (Python >= 3.9). Works on the repo demo photos or your own upload.


## 0 · Setup


In [ ]:
try:
    from plantcv import plantcv as pcv  # noqa
except Exception:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'plantcv', 'opencv-python-headless', 'numpy', 'pandas', 'matplotlib'], check=True)


In [ ]:
import cv2, numpy as np, pandas as pd, os, urllib.request
import matplotlib.pyplot as plt
from plantcv import plantcv as pcv
pcv.params.debug = None
np.set_printoptions(suppress=True, precision=3)
print('PlantCV', pcv.__version__, '| OpenCV', cv2.__version__)


## 1 · Load an image

`medicago_marker.jpg` is a clean, well-exposed marker; `fastplants_colour.jpg` shows two colour morphs.


In [ ]:
IMG  = 'medicago_marker.jpg'
DEMO = 'https://raw.githubusercontent.com/dr-richard-barker/Anthocyanin-Image-analysis/main/public/demos/medicago_marker.jpg'
if not os.path.exists(IMG):
    print('Downloading demo image...'); urllib.request.urlretrieve(DEMO, IMG)
rgb = cv2.cvtColor(cv2.imread(IMG), cv2.COLOR_BGR2RGB)
H, W = rgb.shape[:2]
print('Loaded', IMG, f'{W}x{H}')
plt.figure(figsize=(9,5)); plt.imshow(rgb); plt.title('Input'); plt.axis('off'); plt.show()


## 2 · Detect the Astrocalibration marker

The Astrobotany fiducials are **custom icons** that generic ArUco/AprilTag decoders do not read
reliably, so we find the four dark corner squares **geometrically** (contour-based, dictionary-independent).


In [ ]:
def detect_fiducials(rgb):
    """Find the 4 corner fiducial squares. Returns TL,TR,BR,BL (px) or None."""
    H, W = rgb.shape[:2]
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    th = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 51, 10)
    cnts, _ = cv2.findContours(th, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    cand = []
    for c in cnts:
        a = cv2.contourArea(c)
        if a < W*H*2e-4 or a > W*H*2e-2:
            continue
        ap = cv2.approxPolyDP(c, 0.04*cv2.arcLength(c, True), True)
        if len(ap) != 4 or not cv2.isContourConvex(ap):
            continue
        x, y, w, h = cv2.boundingRect(ap)
        if not (0.7 < w/h < 1.4) or a/(w*h) < 0.6:
            continue
        M = cv2.moments(c); cand.append((a, M['m10']/M['m00'], M['m01']/M['m00']))
    if len(cand) < 4:
        return None
    cand.sort(key=lambda t: -t[0]); med = np.median([c[0] for c in cand[:8]])
    keep = [c for c in cand if 0.5*med < c[0] < 1.8*med]
    pts = np.array([[c[1], c[2]] for c in keep])
    TL = pts[np.argmin(pts[:,0]+pts[:,1])]; BR = pts[np.argmax(pts[:,0]+pts[:,1])]
    TR = pts[np.argmax(pts[:,0]-pts[:,1])]; BL = pts[np.argmin(pts[:,0]-pts[:,1])]
    return np.array([TL, TR, BR, BL], float)

corners = detect_fiducials(rgb)
assert corners is not None, 'No marker found - crop closer to the card.'
print('Fiducial corners (TL,TR,BR,BL):'); print(corners.astype(int))
vis = rgb.copy()
for i, (x, y) in enumerate(corners.astype(int)):
    cv2.circle(vis, (int(x), int(y)), 18, (255,0,255), -1)
    cv2.putText(vis, ['TL','TR','BR','BL'][i], (int(x)+22, int(y)), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255,0,255), 3)
plt.figure(figsize=(9,5)); plt.imshow(vis); plt.title('Detected fiducials'); plt.axis('off'); plt.show()


## 3 · Sample the 15 reference chips

Canonical (landscape) layout: a **colour row** (blue, green, red, yellow, near-black) and a
**grayscale ramp** (white->dark). Chip centres are in normalised quad coords `(u,v)` relative to the
four fiducial centres — the same standard the browser tool uses, so both agree.


In [ ]:
COL = [('blue',0.11,0.50), ('green',0.279,0.50), ('red',0.493,0.50), ('yellow',0.711,0.50), ('near-black',0.89,0.50)]
GRAY_U = np.linspace(0.13, 0.87, 10)
CHIPS = COL + [(f'gray{i}', u, 0.61) for i, u in enumerate(GRAY_U)]

try:
    ASTRO_STD = np.asarray(pcv.transform.astro_color_matrix())[:, 1:]   # 15x3, drop id col
except Exception:
    ASTRO_STD = np.array([[.18,.23,.50],[.34,.62,.25],[.71,.25,.21],[.89,.81,.20],[.21,.22,.22],
                          [.91,.95,.93],[.82,.86,.86],[.72,.75,.73],[.64,.67,.64],[.57,.58,.56],
                          [.48,.49,.48],[.39,.40,.39],[.33,.32,.32],[.27,.28,.27],[.22,.23,.23]])

def quad_pt(C, u, v):
    t = C[0] + (C[1]-C[0])*u; b = C[3] + (C[2]-C[3])*u; return t + (b-t)*v
def sample(rgb, C, u, v, r=8):
    x, y = quad_pt(C, u, v).astype(int); p = rgb[y-r:y+r, x-r:x+r].reshape(-1,3); return np.median(p,0)/255.0

src = np.array([sample(rgb, corners, u, v) for _, u, v in CHIPS])
print('sampled chip RGB (0-255):')
for (n,_,_), s in zip(CHIPS, src):
    print(f'  {n:11} {(s*255).astype(int)}')


## 4 · Affine colour correction

Fit a least-squares 3x4 affine map (sampled chips -> standard) and apply it per pixel — PlantCV's
`affine_color_correction`. The **residual** reports fit quality; a well-exposed, unclipped marker
gives the lowest values.


In [ ]:
try:
    src_pcv = np.hstack([np.arange(len(src)).reshape(-1,1), src])
    tgt_pcv = np.hstack([np.arange(len(src)).reshape(-1,1), ASTRO_STD])
    corrected = pcv.transform.affine_color_correction(rgb_img=rgb, source_matrix=src_pcv, target_matrix=tgt_pcv)
    method = 'PlantCV affine_color_correction'
except Exception as e:
    A = np.hstack([src, np.ones((len(src),1))])
    M, *_ = np.linalg.lstsq(A, ASTRO_STD, rcond=None)
    flat = rgb.reshape(-1,3)/255.0
    corrected = np.clip(np.hstack([flat, np.ones((len(flat),1))]) @ M, 0, 1).reshape(rgb.shape)
    corrected = (corrected*255).astype(np.uint8)
    method = f'least-squares fallback ({type(e).__name__})'

A = np.hstack([src, np.ones((len(src),1))]); M, *_ = np.linalg.lstsq(A, ASTRO_STD, rcond=None)
resid = float(np.sqrt(((A@M - ASTRO_STD)**2).sum(1).mean()))
print(f'Colour correction via {method}  |  RMS residual = {resid:.3f}')
fig, ax = plt.subplots(1, 2, figsize=(13,5))
ax[0].imshow(rgb); ax[0].set_title('Original'); ax[0].axis('off')
ax[1].imshow(corrected); ax[1].set_title('Colour-corrected'); ax[1].axis('off'); plt.show()


## 5 · Scale (pixels -> cm)

Adjust `MARKER_SPAN_CM` to the centre-to-centre distance of your marker's corner fiducials.


In [ ]:
MARKER_SPAN_CM = 4.3
top = np.linalg.norm(corners[1]-corners[0]); left = np.linalg.norm(corners[3]-corners[0])
px_per_cm = ((top+left)/2)/MARKER_SPAN_CM
print(f'Resolution ~ {px_per_cm:.1f} px/cm  ({1/px_per_cm:.4f} cm/px)')


## 6 · Segment the plant (Excess-Green)

ExG = 2G - R - B on the corrected image, thresholded to isolate vegetation; the marker area is masked out.


In [ ]:
f = corrected.astype(np.float32)
exg = 2*f[:,:,1] - f[:,:,0] - f[:,:,2]
EXG_THRESH = 25
mask = (exg > EXG_THRESH).astype(np.uint8)*255
x0, y0 = corners.min(0).astype(int); x1, y1 = corners.max(0).astype(int)
pad = int(0.15*max(x1-x0, y1-y0))
mask[max(0,y0-pad):y1+pad, max(0,x0-pad):x1+pad] = 0
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))
plt.figure(figsize=(9,5)); plt.imshow(mask, cmap='Greens'); plt.title(f'Plant mask (ExG>{EXG_THRESH})'); plt.axis('off'); plt.show()
print('plant pixels:', int((mask>0).sum()))


## 7 · Quantify: leaf area + indices

On the **colour-corrected** pixels inside the mask. **NGRDI**=(G-R)/(G+R) · **mACI**=R/G · **GI**=G/(R+G+B).


In [ ]:
m = mask > 0
R, G, B = [corrected[:,:,i].astype(np.float32)[m] for i in range(3)]
eps = 1e-6
ngrdi = float(np.mean((G-R)/(G+R+eps))); maci = float(np.mean(R/(G+eps))); gi = float(np.mean(G/(R+G+B+eps)))
area_cm2 = float(m.sum()/(px_per_cm**2))
summary = pd.DataFrame([{ 'image': os.path.basename(IMG), 'plant_px': int(m.sum()), 'area_cm2': round(area_cm2,3),
    'NGRDI': round(ngrdi,4), 'mACI': round(maci,4), 'GI': round(gi,4),
    'colour_residual': round(resid,3), 'px_per_cm': round(px_per_cm,1)}])
summary


## 8 · Export

For a **batch / timelapse**, loop cells 1–7 over a folder sharing one marker and `pd.concat` the summaries.


In [ ]:
stem = os.path.splitext(os.path.basename(IMG))[0]
cv2.imwrite(f'{stem}_corrected.png', cv2.cvtColor(corrected, cv2.COLOR_RGB2BGR))
summary.to_csv(f'{stem}_phenotype.csv', index=False)
print('wrote', f'{stem}_corrected.png', 'and', f'{stem}_phenotype.csv')


---
### Notes
- **Why a notebook as well as the browser tool?** Generic ArUco/AprilTag libraries do not decode the
  Astrobotany custom fiducials reliably, and robust colour-card + chip detection is far easier with
  OpenCV/PlantCV than in-browser. The notebook is the dependable *batch/research* path; the browser tool
  is the accessible *interactive* path. Both share the same 15-chip standard and index definitions.
- **Get a marker:** <https://www.astrobotany.com> (AIRI Bio-Imaging Spectrum 5-cm sticker).
- **Calibrate mACI->anthocyanin** against a pigment assay for absolute values; the index alone is a relative proxy.
